# Directional horizontal connections — the envelope in 3D

Every plot here is a **live plotly figure**: drag to rotate, scroll to zoom, double-click to reset.

**The mechanism, in one line.** Each unit `p` on the feature map gets its own copy of the shared
lateral kernel, re-weighted by an envelope that rises along that unit's own outward radius:

```
lat_c(p) = N(c,p) * sum_q { W_c(q) * E_p(q) * h_c(p+q) }

  E_p(q) = exp( vec(p) . q )     vec(p) = beta(r_p) * rhat_p = (a(p), b(p))
  N(c,p) = sum_q |W_c(q)| / sum_q |W_c(q) E_p(q)|
```

`q` is a tap offset inside the 11x11 window. `rhat_p` is the outward radial unit vector at `p`.
`beta` is a **log-slope in 1/px**: step one pixel along `vec` and the envelope multiplies by
`exp(beta)`.

### Running it

- **Google Colab** — just run every cell. Nothing to install; the notebook is self-contained.
  Optionally upload a checkpoint in cell 2 to use the real trained kernels.
- **Inside the repo** — it detects `src.models.utils.radial_shift`, uses the real module, and
  **cross-checks the inlined formula against it**.

In [ ]:
# ============================== CONFIG ==============================
# Paste a run's own "[dr]" line. This is the only thing you need to reproduce a run's geometry.
LOG_LINE  = ("[dr] ep49 stages.0.0  w=[+0.36 -0.74 -0.13]  c=[16.4 34.1 43.5]px  "
             "s=[1.14 1.00 2.61]px  b=+0.469")
BETA_MAX  = 0.640          # MUST match config.model.directional_beta_max for that run
TAPER_PX  = 3.0            # RadialShift's taper_px
HW        = (156, 156)     # stage-0 feature map
KSIZE     = 11             # lateral kernel size
CKPT_PATH = ("/home/tomasdu/repos/experiments/plastic_NNs/active/WS-S/wandb/"
             "offline-run-20260908_223210-41xkud1m/files/model/epoch_0029.pth")  # "" to skip
ZONES = dict(core=23.84, rfov=29.88, footprint=41.70)   # measured, feature-map px
# ====================================================================

import math, os, re, sys
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

try:                                   # Colab renders plotly inline without help; be explicit anyway
    import google.colab                # noqa
    pio.renderers.default = "colab"
    IN_COLAB = True
except Exception:
    IN_COLAB = False

TPL   = "plotly_white"
CDIV  = "RdBu_r"          # signed quantities (kernels)
CPOS  = "Plasma"          # positive quantities (the envelope)

# ---------------------------------------------------------------------------------------------
# beta(r), TRANSCRIBED from src/models/utils/radial_shift.py (RadialShift.beta_at).
# This copy exists ONLY so the notebook runs on Colab with no repo. It is a duplicate and can
# therefore DRIFT: when the repo is importable, the cell below asserts the two agree, and prints
# the max deviation. Trust that check, not this comment.
# ---------------------------------------------------------------------------------------------
def beta_at(r, w, c, s, b, beta_max=BETA_MAX, taper_px=TAPER_PX):
    r = np.asarray(r, dtype=float)
    z = float(b) + sum(wk / (1.0 + np.exp(-(r - ck) / sk)) for wk, ck, sk in zip(w, c, s))
    return beta_max * np.tanh(z) * (1.0 - np.exp(-(r / taper_px) ** 2))

def parse_log(line):
    v = lambda t: [float(x) for x in re.search(rf"\b{t}=\[([^\]]*)\]", line).group(1).split()]
    mb = re.search(r"\bb=([-+]?[0-9.]+)", line)
    if mb is None:
        raise ValueError("no 'b=' in the log line -- the fovea baseline is not optional")
    return v("w"), v("c"), v("s"), float(mb.group(1))

def gabor(k, theta, lam=5.32, sigma=2.2, phase=0.0):
    """A zero-DC Gabor. Stand-in for a trained lateral kernel when no checkpoint is supplied."""
    a = np.arange(k) - (k - 1) / 2.0
    X, Y = np.meshgrid(a, a)
    xr = X * np.cos(theta) + Y * np.sin(theta)
    yr = -X * np.sin(theta) + Y * np.cos(theta)
    g = np.exp(-(xr ** 2 + yr ** 2) / (2 * sigma ** 2)) * np.cos(2 * np.pi * xr / lam + phase)
    return g - g.mean()

W_PAR, C_PAR, S_PAR, B_PAR = parse_log(LOG_LINE)

# ---- kernels: the checkpoint's if we can get them, else a Gabor bank ----
KERNELS, KSRC, RAW_WIDTH = None, "", None
if CKPT_PATH and os.path.isfile(CKPT_PATH):
    import torch
    ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    sd = ck.get("model_state_dict", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck
    KERNELS = sd["stages.0.0.lateral.weight"].float().numpy().reshape(-1, KSIZE, KSIZE)
    sub = {k.split("lateral_shift.")[1]: v.float().numpy()
           for k, v in sd.items() if "lateral_shift." in k}
    if sub:                                   # prefer the checkpoint's OWN parameters
        W_PAR = list(sub["weight"][0]); C_PAR = list(sub["centre"][0])
        # widths() is softplus(_width) + 1e-3, so this is the width in px.
        S_PAR = list(np.log1p(np.exp(sub["_width"][0])) + 1e-3); B_PAR = float(sub["bias"][0][0])
        RAW_WIDTH = sub["_width"][0]          # keep the RAW parameter for the cross-check
    KSRC = f"checkpoint {os.path.basename(CKPT_PATH)}"
else:
    KERNELS = np.stack([gabor(KSIZE, t) for t in np.linspace(0, math.pi, 32, endpoint=False)])
    KSRC = "a synthetic Gabor bank (no checkpoint)"

NCH  = KERNELS.shape[0]
PAD  = KSIZE // 2
QQ   = np.arange(KSIZE) - PAD
QX, QY = np.meshgrid(QQ, QQ)                       # QX[i,j]=q_x, QY[i,j]=q_y, rows=y cols=x
CY   = (HW[0] - 1) / 2.0
yy, xx = np.mgrid[0:HW[0], 0:HW[1]]
DY, DX = yy - CY, xx - CY
RMAP = np.hypot(DY, DX)
RSAFE = np.maximum(RMAP, 1e-6)
COS_T, SIN_T = DX / RSAFE, DY / RSAFE               # rhat, split into components
BETA_MAP = beta_at(RMAP, W_PAR, C_PAR, S_PAR, B_PAR)
A_MAP, B_MAP = BETA_MAP * COS_T, BETA_MAP * SIN_T   # the polarity vector field

def envelope(p):
    """E_p(q) over the whole (k,k) window for the unit at p=(y,x)."""
    return np.exp(A_MAP[p] * QX + B_MAP[p] * QY)

def com(M):
    """|w|-centre of mass, (dy,dx) in px from the centre tap."""
    m = np.abs(M); t = m.sum()
    return float((m.sum(1) * QQ).sum() / t), float((m.sum(0) * QQ).sum() / t)

def l1(M, base):
    return M * (base / np.abs(M).sum())

def at_ecc(r, deg):
    return (int(round(CY + r * math.sin(math.radians(deg)))),
            int(round(CY + r * math.cos(math.radians(deg)))))

# ---- cross-check the inlined formula against the shipped module, when it is importable ----
REPO = "/exports/home/tomasdu/repos/plastic_NNs"
CHECK = "not run (repo not importable -- expected on Colab)"
try:
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    import torch
    from src.models.utils.radial_shift import RadialShift
    _rs = RadialShift(input_size=HW, k_steps=len(W_PAR), beta_max=BETA_MAX, taper_px=TAPER_PX)
    with torch.no_grad():
        _rs.weight[0] = torch.tensor(W_PAR); _rs.centre[0] = torch.tensor(C_PAR)
        _rs.bias[0, 0] = B_PAR
        # INVERTING widths() CORRECTLY. widths() = softplus(_width) + 1e-3, so the inverse is
        # log(expm1(s - 1e-3)), NOT log(expm1(s)) -- the latter applies the epsilon a second time
        # and every width comes back 0.001 px too wide. Tiny in effect, but it puts a ~7e-5 noise
        # floor under this check, which then cannot detect the drift it exists to detect.
        # Better still: when the checkpoint gave us the raw parameter, skip the round trip entirely.
        _rs._width[0] = (torch.as_tensor(RAW_WIDTH) if RAW_WIDTH is not None
                         else torch.log(torch.expm1(torch.tensor(S_PAR) - 1e-3)))
        _ref = _rs.beta()[0].numpy()
    CHECK = (f"max |inlined - shipped RadialShift| = {np.abs(_ref - BETA_MAP).max():.3e}"
             f"  ({'raw _width' if RAW_WIDTH is not None else 'inverted from s'})")
except Exception as e:
    CHECK = f"not run ({type(e).__name__}) -- expected on Colab"

print(f"kernels   : {KERNELS.shape}  from {KSRC}")
print(f"beta      : w={np.round(W_PAR,3).tolist()}  c={np.round(C_PAR,1).tolist()}  "
      f"s={np.round(S_PAR,2).tolist()}  b={B_PAR:+.3f}   beta_max={BETA_MAX}")
print(f"            over the map: {BETA_MAP.min():+.4f} .. {BETA_MAP.max():+.4f} 1/px")
print(f"formula check: {CHECK}")

---
## 1 — `beta(r)`, as a surface over the feature map

`beta` depends on eccentricity only, so revolving the learned profile gives a surface with circular
symmetry. **Height and colour are both `beta`.** The rings mark the measured zones.

Rotate to a low angle to read it as the profile; look from above to see it as a map.

In [ ]:
ST = 2
Zb = BETA_MAP[::ST, ::ST]
fig = go.Figure()
fig.add_trace(go.Surface(
    z=Zb, x=xx[0, ::ST], y=yy[::ST, 0], colorscale=CDIV, cmid=0,
    colorbar=dict(title="beta<br>(1/px)", len=0.7),
    contours={"z": {"show": True, "start": -0.05, "end": 0.4, "size": 0.05,
                    "color": "rgba(0,0,0,0.35)", "project": {"z": True}}},
    hovertemplate="x=%{x}<br>y=%{y}<br>beta=%{z:.4f}<extra></extra>"))
for r0, nm, col in ((ZONES["core"], "occluded core", "black"),
                    (ZONES["rfov"], "fisheye rfov", "dimgray"),
                    (ZONES["footprint"], "scotoma footprint", "gray")):
    t = np.linspace(0, 2*np.pi, 240)
    fig.add_trace(go.Scatter3d(x=CY + r0*np.cos(t), y=CY + r0*np.sin(t),
                               z=beta_at(np.full_like(t, r0), W_PAR, C_PAR, S_PAR, B_PAR) + 0.004,
                               mode="lines", line=dict(width=6, color=col), name=f"{nm} ({r0:.1f}px)"))
fig.update_layout(
    template=TPL, height=680,
    title=dict(text="<b>1 — the learned polarity beta(r), revolved over the feature map</b><br>"
                    "<sup>height = colour = beta (1/px). Positive = that unit reads FURTHER OUT. "
                    "Circular by construction: beta depends on eccentricity only.</sup>"),
    scene=dict(xaxis_title="x (feature-map px)", yaxis_title="y (feature-map px)",
               zaxis_title="beta (1/px)", aspectratio=dict(x=1, y=1, z=0.45),
               camera=dict(eye=dict(x=1.5, y=-1.5, z=0.9))),
    legend=dict(orientation="h", y=-0.02))
fig.show()

---
## 2 — the field `vec(p) = beta(r_p) * rhat_p`

One arrow per position: direction `rhat_p` (straight out from the fovea), length `|beta(r_p)|`.
Where `beta < 0` the cone flips and points at the fovea.

This is still only a field of numbers — nothing is "aimed" until it multiplies a kernel in Part 3.

In [ ]:
ST = 6
sy, sx = np.mgrid[3:HW[0]:ST, 3:HW[1]:ST]
u, v = A_MAP[3::ST, 3::ST], B_MAP[3::ST, 3::ST]
bb   = BETA_MAP[3::ST, 3::ST]
fig = go.Figure(go.Cone(
    x=sx.ravel(), y=sy.ravel(), z=np.zeros(sx.size),
    u=u.ravel(), v=v.ravel(), w=np.zeros(u.size),
    anchor="tail", sizemode="scaled", sizeref=14.0,
    colorscale=CDIV, cmid=0, cmin=-BETA_MAX, cmax=BETA_MAX,
    colorbar=dict(title="beta<br>(1/px)", len=0.7),
    hovertemplate="x=%{x}<br>y=%{y}<br>|vec|=%{u:.3f}<extra></extra>"))
for r0, col in ((ZONES["core"], "black"), (ZONES["rfov"], "dimgray"),
                (ZONES["footprint"], "gray")):
    t = np.linspace(0, 2*np.pi, 240)
    fig.add_trace(go.Scatter3d(x=CY + r0*np.cos(t), y=CY + r0*np.sin(t), z=np.zeros_like(t),
                               mode="lines", line=dict(width=4, color=col),
                               name=f"r = {r0:.1f} px"))
fig.update_layout(
    template=TPL, height=680,
    title=dict(text="<b>2 — the polarity vector field vec(p) = beta(r_p) * rhat_p</b><br>"
                    "<sup>direction = straight out from the fovea; length = |beta|. "
                    "Look from directly above for the cleanest read.</sup>"),
    scene=dict(xaxis_title="x (feature-map px)", yaxis_title="y (feature-map px)",
               zaxis=dict(title="", showticklabels=False, range=[-8, 8]),
               aspectratio=dict(x=1, y=1, z=0.18),
               camera=dict(eye=dict(x=0.1, y=-0.1, z=2.1))),
    legend=dict(orientation="h", y=-0.02))
fig.show()

_inner = (RMAP > 2) & (RMAP < 45)
_dot = (A_MAP * COS_T + B_MAP * SIN_T)[_inner]
print(f"self-check: {100*(_dot > 0).mean():.1f}% of the {_inner.sum():,} positions with "
      f"2 < r < 45 have vec . rhat > 0  ->  all pointing OUTWARD")

---
## 3 — the arrow becomes a **plane**, and the plane becomes a ramp

For a unit at `p`, take logs of the envelope:

```
log E_p(q) = vec(p) . q  =  a(p)*q_x + b(p)*q_y
```

That is **linear in `q`** — no squares, no cross terms — so graphed as a height over the 11x11
window it is a **flat tilted sheet through the origin**.

- **gradient** = `(d/dq_x, d/dq_y) = (a, b) = vec(p)`: the direction of steepest climb, and the rise
  per pixel is `|vec| = |beta|`.
- **level sets** = the lines where the height is constant. For a plane those are **straight parallel
  lines perpendicular to `vec`** — projected onto the floor of the left panel.
- the sheet passes through **0 at the centre tap**, so `E = exp(0) = 1` there. Always.

The right panel is `exp` of the left. `exp` bends the flat sheet into a ramp, turns equal *steps*
into equal *ratios*, and leaves the level sets exactly where they were.

**Rotate both.** They are locked to the same camera.

In [ ]:
P = at_ecc(24.5, 57)
ap, bp = A_MAP[P], B_MAP[P]
beta_p = BETA_MAP[P]
th_p   = math.degrees(math.atan2(SIN_T[P], COS_T[P]))
logE   = ap * QX + bp * QY
E      = np.exp(logE)
n      = math.hypot(ap, bp); ux, uy = ap/n, bp/n

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "surface"}, {"type": "surface"}]],
                    subplot_titles=("log E  =  vec . q      (a PLANE)",
                                    "E  =  exp(vec . q)      (a RAMP)"),
                    horizontal_spacing=0.03)
fig.add_trace(go.Surface(
    z=logE, x=QQ, y=QQ, colorscale=CDIV, cmid=0, showscale=False,
    contours={"z": {"show": True, "size": 0.5, "color": "rgba(0,0,0,0.5)",
                    "project": {"z": True}}},
    hovertemplate="q_x=%{x}<br>q_y=%{y}<br>log E=%{z:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Surface(
    z=E, x=QQ, y=QQ, colorscale=CPOS, showscale=True,
    colorbar=dict(title="E", len=0.65, x=1.02),
    contours={"z": {"show": True, "size": 2.0, "color": "rgba(0,0,0,0.45)"}},
    hovertemplate="q_x=%{x}<br>q_y=%{y}<br>E=%{z:.3f}<extra></extra>"), row=1, col=2)
# the gradient, drawn ON the plane
tt = np.linspace(0, 5, 30)
fig.add_trace(go.Scatter3d(x=ux*tt, y=uy*tt, z=(ap*ux + bp*uy)*tt, mode="lines",
                           line=dict(width=9, color="black"), name="vec(p) (steepest climb)"),
              row=1, col=1)
# a level set through the centre: perpendicular to vec, height stays 0
pp = np.linspace(-5, 5, 30)
fig.add_trace(go.Scatter3d(x=-uy*pp, y=ux*pp, z=np.zeros_like(pp), mode="lines",
                           line=dict(width=9, color="limegreen"),
                           name="a level set (perpendicular, height constant)"), row=1, col=1)
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
                           marker=dict(size=6, color="black"),
                           name="centre tap: log E = 0, E = 1"), row=1, col=1)

cam = dict(eye=dict(x=1.6, y=-1.6, z=1.0))
axes = dict(xaxis_title="q_x (px)", yaxis_title="q_y (px)")
fig.update_layout(
    template=TPL, height=680,
    title=dict(text=f"<b>3 — the envelope at one unit: p={P}, r={RMAP[P]:.1f}px, "
                    f"theta={th_p:+.0f}deg, beta={beta_p:+.4f} 1/px</b><br>"
                    f"<sup>left: the PLANE, with its floor contours = the level sets. "
                    f"right: exp of it. One px along vec multiplies E by "
                    f"exp(beta) = {math.exp(beta_p):.4f}.</sup>"),
    scene =dict(**axes, zaxis_title="log E", camera=cam, aspectratio=dict(x=1, y=1, z=0.7)),
    scene2=dict(**axes, zaxis_title="E",     camera=cam, aspectratio=dict(x=1, y=1, z=0.7)),
    legend=dict(orientation="h", y=-0.02))
fig.show()

print(f"E at the centre tap        : {E[PAD,PAD]:.6f}")
print(f"one px ALONG vec           : x{math.exp(beta_p):.4f}   (constant, every px)")
print(f"one px ACROSS vec          : x{math.exp(0):.4f}")
print(f"E range over the window    : {E.min():.4f} .. {E.max():.4f}   "
      f"(ratio {E.max()/E.min():.1f}x)")
print(f"log E range                : {logE.min():+.4f} .. {logE.max():+.4f}   "
      f"(symmetric: {logE.min():+.4f} = -{logE.max():.4f})")

---
## 4 — sweep `beta`: watch the plane tilt

Same unit, same direction, `beta` swept from `-beta_max` to `+beta_max`. Drag the slider.

At `beta = 0` the plane is **exactly flat** and `E == 1` everywhere — the envelope is the identity,
and the block is bit-identical to a plain `conv2d`. That is what the module initialises to, so "no
polarity learned" is a reachable answer rather than one the parametrisation forbids.

In [ ]:
BETAS = np.round(np.linspace(-BETA_MAX, BETA_MAX, 17), 3)
proj  = COS_T[P] * QX + SIN_T[P] * QY          # q . rhat_p, fixed; only beta changes
frames, zmax = [], math.exp(BETA_MAX * abs(proj).max())
for bta in BETAS:
    frames.append(go.Frame(name=f"{bta:+.3f}", data=[
        go.Surface(z=bta*proj, x=QQ, y=QQ, colorscale=CDIV, cmid=0, cmin=-2.7, cmax=2.7,
                   showscale=False,
                   contours={"z": {"show": True, "size": 0.5, "color": "rgba(0,0,0,0.5)",
                                   "project": {"z": True}}}),
        go.Surface(z=np.exp(bta*proj), x=QQ, y=QQ, colorscale=CPOS,
                   cmin=0, cmax=zmax, showscale=False)]))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "surface"}, {"type": "surface"}]],
                    subplot_titles=("log E — the plane tilts", "E — the ramp steepens"),
                    horizontal_spacing=0.03)
fig.add_trace(frames[len(BETAS)//2].data[0], row=1, col=1)
fig.add_trace(frames[len(BETAS)//2].data[1], row=1, col=2)
fig.frames = frames
cam  = dict(eye=dict(x=1.6, y=-1.6, z=1.0))
axes = dict(xaxis_title="q_x (px)", yaxis_title="q_y (px)")
fig.update_layout(
    template=TPL, height=700,
    title=dict(text="<b>4 — sweeping beta at a fixed direction</b><br>"
                    "<sup>beta = 0 gives a FLAT plane and E == 1 everywhere: the envelope is "
                    "exactly the identity. Sign flips which way the ramp leans.</sup>"),
    scene =dict(**axes, zaxis=dict(title="log E", range=[-2.9, 2.9]), camera=cam,
                aspectratio=dict(x=1, y=1, z=0.7)),
    scene2=dict(**axes, zaxis=dict(title="E", range=[0, zmax]), camera=cam,
                aspectratio=dict(x=1, y=1, z=0.7)),
    sliders=[dict(active=len(BETAS)//2, y=0, x=0.08, len=0.85,
                  currentvalue=dict(prefix="beta = ", suffix=" 1/px", font=dict(size=15)),
                  steps=[dict(method="animate", label=f"{b:+.3f}",
                              args=[[f"{b:+.3f}"], dict(mode="immediate",
                                    frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
                         for b in BETAS])])
fig.show()

---
## 5 — what the ramp does to a real kernel

Three surfaces for one channel at one unit:

1. `W_c` — the trained kernel, signed. Blue and red lobes.
2. `E_p` — the ramp (identical for every channel; only `W_c` carries the channel index).
3. `W_c * E_p * N` — the product, L1-renormalised.

Two things to look for while rotating: **no lobe changes sign** — the envelope is strictly positive,
so it rescales and never flips — and the **`|w|` centre of mass slides along `vec`**, marked by the
black stems.

In [ ]:
CH = 0
Wc  = KERNELS[CH]
E   = envelope(P)
eff = l1(Wc * E, np.abs(Wc).sum())
cy0, cx0 = com(Wc); cy1, cx1 = com(eff)
N   = np.abs(Wc).sum() / np.abs(Wc * E).sum()
vmax = float(np.abs(Wc).max())

fig = make_subplots(rows=1, cols=3,
                    specs=[[{"type": "surface"}]*3],
                    subplot_titles=(f"W_c  (channel {CH})", "E_p  (same for every channel)",
                                    "W_c * E_p * N"), horizontal_spacing=0.02)
fig.add_trace(go.Surface(z=Wc, x=QQ, y=QQ, colorscale=CDIV, cmid=0, showscale=False,
                         hovertemplate="q=(%{x},%{y})<br>W=%{z:.4f}<extra></extra>"), 1, 1)
fig.add_trace(go.Surface(z=E, x=QQ, y=QQ, colorscale=CPOS, showscale=False,
                         hovertemplate="q=(%{x},%{y})<br>E=%{z:.3f}<extra></extra>"), 1, 2)
fig.add_trace(go.Surface(z=eff, x=QQ, y=QQ, colorscale=CDIV, cmid=0, showscale=False,
                         hovertemplate="q=(%{x},%{y})<br>W.E.N=%{z:.4f}<extra></extra>"), 1, 3)
for col, (cy, cx, zt) in ((1, (cy0, cx0, vmax)), (3, (cy1, cx1, vmax))):
    fig.add_trace(go.Scatter3d(x=[cx, cx], y=[cy, cy], z=[-zt, zt], mode="lines+markers",
                               line=dict(width=8, color="black"),
                               marker=dict(size=4, color="black"),
                               name=f"|w| centre of mass ({'before' if col==1 else 'after'})"),
                  1, col)
tt = np.linspace(0, 4.2, 20)
n_ = math.hypot(ap, bp)
for col in (1, 3):
    fig.add_trace(go.Scatter3d(x=ap/n_*tt, y=bp/n_*tt, z=np.zeros_like(tt), mode="lines",
                               line=dict(width=7, color="limegreen"), showlegend=(col == 1),
                               name="vec(p)"), 1, col)
cam = dict(eye=dict(x=1.5, y=-1.5, z=1.2))
sc  = dict(xaxis_title="q_x", yaxis_title="q_y", camera=cam, aspectratio=dict(x=1, y=1, z=0.6))
along = (cx1-cx0)*ap/n_ + (cy1-cy0)*bp/n_
fig.update_layout(
    template=TPL, height=620,
    title=dict(text=f"<b>5 — the ramp re-weights the kernel</b>   "
                    f"p={P}, beta={beta_p:+.4f}, N={N:.4f}<br>"
                    f"<sup>centre of mass ({cy0:+.2f},{cx0:+.2f}) -> ({cy1:+.2f},{cx1:+.2f}), "
                    f"i.e. {along:+.2f} px along vec. Signs preserved: "
                    f"{bool(np.all((Wc > 0) == (eff > 0)))}.</sup>"),
    scene=dict(**sc, zaxis_title="w"), scene2=dict(**sc, zaxis_title="E"),
    scene3=dict(**sc, zaxis_title="w"), legend=dict(orientation="h", y=-0.02))
fig.show()

print(f"  {'ch':>3}{'COM before':>18}{'COM after':>18}{'along vec':>11}{'perp':>8}"
      f"{'N':>8}{'signs':>8}")
for c in range(min(6, NCH)):
    Wc_ = KERNELS[c]; ef = l1(Wc_ * E, np.abs(Wc_).sum())
    y0, x0 = com(Wc_); y1, x1 = com(ef)
    dvx, dvy = x1-x0, y1-y0
    al = dvx*ap/n_ + dvy*bp/n_
    pe = math.sqrt(max(dvx**2 + dvy**2 - al**2, 0))
    print(f"  {c:>3}   ({y0:+.2f},{x0:+.2f})   ({y1:+.2f},{x1:+.2f}){al:>11.2f}{pe:>8.2f}"
          f"{np.abs(Wc_).sum()/np.abs(Wc_*E).sum():>8.4f}"
          f"{'kept' if np.all((Wc_>0)==(ef>0)) else 'FLIPPED':>8}")

---
## 6 — walk out along a meridian

The same channel's kernel at increasing eccentricity, following the **learned** profile. Drag the
slider from the fovea outward and watch the ramp switch on, saturate through the LPZ, and fade back
toward flat past the scotoma footprint.

In [ ]:
ECCS = np.arange(0, 61, 4)
DEG  = 57
Wc   = KERNELS[CH]; base = np.abs(Wc).sum()
frames, mx = [], 0.0
for r0 in ECCS:
    pp_ = at_ecc(r0, DEG)
    ef  = l1(Wc * envelope(pp_), base)
    mx  = max(mx, float(np.abs(ef).max()))
    frames.append((r0, pp_, ef))
zr = [-mx, mx]
gframes = []
for r0, pp_, ef in frames:
    gframes.append(go.Frame(name=f"{r0}", data=[
        go.Surface(z=ef, x=QQ, y=QQ, colorscale=CDIV, cmid=0, cmin=-mx, cmax=mx, showscale=False)]))
fig = go.Figure(data=[gframes[0].data[0]], frames=gframes)
fig.update_layout(
    template=TPL, height=660,
    title=dict(text=f"<b>6 — the effective kernel along a meridian (theta={DEG}deg, "
                    f"channel {CH})</b><br>"
                    "<sup>L1-renormalised at every step, so what changes is the SHAPE, not the "
                    "total weight. Flat at the fovea (taper), strongest through the LPZ.</sup>"),
    scene=dict(xaxis_title="q_x (px)", yaxis_title="q_y (px)",
               zaxis=dict(title="w", range=zr), aspectratio=dict(x=1, y=1, z=0.6),
               camera=dict(eye=dict(x=1.5, y=-1.5, z=1.1))),
    sliders=[dict(active=0, y=0, x=0.08, len=0.85,
                  currentvalue=dict(prefix="eccentricity r = ", suffix=" px", font=dict(size=15)),
                  steps=[dict(method="animate", label=f"{r0}",
                              args=[[f"{r0}"], dict(mode="immediate",
                                    frame=dict(duration=0, redraw=True),
                                    transition=dict(duration=0))]) for r0, _, _ in frames])])
fig.show()

print(f"  {'r (px)':>7}{'beta':>10}{'E min':>9}{'E max':>9}{'COM along vec':>15}")
for r0, pp_, ef in frames[::2]:
    E_ = envelope(pp_); y1, x1 = com(l1(Wc*E_, base)); y0, x0 = com(Wc)
    nn = math.hypot(A_MAP[pp_], B_MAP[pp_]) or 1.0
    al = (x1-x0)*A_MAP[pp_]/nn + (y1-y0)*B_MAP[pp_]/nn
    print(f"  {r0:>7}{BETA_MAP[pp_]:>+10.4f}{E_.min():>9.3f}{E_.max():>9.2f}{al:>15.2f}")

---
## 7 — the achieved displacement, over the whole map

`beta` is one number per eccentricity; the **pixels** it buys are not. They depend on each channel's
own structure and on the angle between that kernel's carrier and the radius.

**Height and colour** = how far the `|w|` centre of mass actually moved, radially, median over all
channels. This is the retinotopic field of receptive-field displacements the 10 learned parameters
produce.

In [ ]:
ST = 5
py, px = np.mgrid[6:HW[0]-5:ST, 6:HW[1]-5:ST]
pos = np.stack([py.ravel(), px.ravel()], 1)
av, bv = A_MAP[pos[:,0], pos[:,1]], B_MAP[pos[:,0], pos[:,1]]
E_all = np.exp(av[:,None,None]*QX[None] + bv[:,None,None]*QY[None])       # (P,k,k)
m  = np.abs(KERNELS[None] * E_all[:,None])                               # (P,C,k,k)
t  = np.maximum(m.sum((-1,-2)), 1e-12)
cyc = (m.sum(-1) * QQ).sum(-1) / t
cxc = (m.sum(-2) * QQ).sum(-1) / t
nn  = np.maximum(np.hypot(av, bv), 1e-12)
# project PER CHANNEL then take the median -- projecting the channel-median centroid instead
# cancels the per-channel anisotropy against itself and hides the off-radial residual.
along_c = cxc*(av/nn)[:,None] + cyc*(bv/nn)[:,None]
perp_c  = np.sqrt(np.maximum(cxc**2 + cyc**2 - along_c**2, 0))
along, perp = np.median(along_c, 1), np.median(perp_c, 1)
Zd = along.reshape(py.shape)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "surface"}, {"type": "surface"}]],
                    subplot_titles=("radial displacement (px)",
                                    "off-radial residual (px)"), horizontal_spacing=0.04)
fig.add_trace(go.Surface(z=Zd, x=px[0], y=py[:,0], colorscale=CDIV, cmid=0,
                         colorbar=dict(title="px", len=0.6, x=0.44),
                         contours={"z": {"show": True, "size": 0.5,
                                         "color": "rgba(0,0,0,0.35)", "project": {"z": True}}},
                         hovertemplate="x=%{x}<br>y=%{y}<br>%{z:.2f} px<extra></extra>"), 1, 1)
fig.add_trace(go.Surface(z=perp.reshape(py.shape), x=px[0], y=py[:,0], colorscale="Viridis",
                         colorbar=dict(title="px", len=0.6, x=1.01),
                         hovertemplate="x=%{x}<br>y=%{y}<br>%{z:.3f} px<extra></extra>"), 1, 2)
cam = dict(eye=dict(x=1.5, y=-1.5, z=1.0))
fig.update_layout(
    template=TPL, height=650,
    title=dict(text="<b>7 — how far the receptive field actually moves</b><br>"
                    "<sup>median over channels. Left: along the radius (the intended effect). "
                    "Right: perpendicular to it — the kernels' own anisotropy, "
                    "which the envelope does not control.</sup>"),
    scene =dict(xaxis_title="x (px)", yaxis_title="y (px)", zaxis_title="px along vec",
                camera=cam, aspectratio=dict(x=1, y=1, z=0.5)),
    scene2=dict(xaxis_title="x (px)", yaxis_title="y (px)", zaxis_title="px perpendicular",
                camera=cam, aspectratio=dict(x=1, y=1, z=0.5)))
fig.show()

inner = np.hypot(pos[:,0]-CY, pos[:,1]-CY) < ZONES["footprint"]
print(f"inside the scotoma footprint (r < {ZONES['footprint']:.0f} px, {int(inner.sum())} positions):")
print(f"  radial displacement : median {np.median(along[inner]):+.2f} px, "
      f"max {along[inner].max():+.2f}")
print(f"  off-radial residual : median {np.median(perp[inner]):.2f} px, "
      f"max {perp[inner].max():.2f}  "
      f"({100*np.median(perp[inner])/max(np.median(np.abs(along[inner])),1e-9):.0f}% of the radial move)")
print(f"\nThe far field is flat because beta ~ 0 there, not because the operator is weak.")

---
## Recap

| | |
|---|---|
| `beta(r)` | 10 learned parameters -> one log-slope per eccentricity, in 1/px |
| `vec(p) = beta * rhat` | a vector field: which way out, how steeply |
| `log E_p(q) = vec . q` | a **plane** over the kernel window; gradient `vec`, level sets perpendicular |
| `E_p(q) = exp(vec . q)` | a positive ramp: `exp(beta)` per px along `vec`, exactly 1 at the centre tap |
| `W_c * E_p` | signs untouched, magnitudes re-weighted, centre of mass slides outward |
| `* N(c,p)` | pins `sum|w|`, so `beta` sets **direction** and `lateral_gate` still owns **magnitude** |

The thing to carry away: `beta` is a **slope**, not a displacement. The displacement is the enveloped
kernel's centre of mass, it depends on the kernel, and it saturates against the `(k-1)/2 = 5 px`
reach of an 11x11 window. Figure 7's right panel is the part the envelope cannot control.